# Física de Neutrinos en el Espacio qang: Simulación y Auditoría de Falsación (Parte VII)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_neutrinos_es.ipynb)

Este cuaderno implementa de forma interactiva los experimentos de **falsación y dinámica de neutrinos** integrados en la obra *Computación, información y física cuántica en el espacio qang* (35 capítulos, Parte VII: Evidencia experimental y auditoría de falsación).

Un neutrino se produce y detecta en un autoestado de sabor ($e, \mu, \tau$) pero se propaga como superposición de estados de masa. En la representación geométrica $qg$, esta oscilación es una rotación paramétrica dentro de un subespacio de peso de Hamming conservado.

Alineado con las especificaciones de **`qang 0.14.0`** y su módulo `qang.integrations`, este cuaderno ejecuta las simulaciones en backends de ruido calibrado de IBM (`FakeBrisbane`, `FakeTorino`), comparando la medición sin mitigación (*raw shots*) frente a la proyección filtrada por simetría $qg$ (*weight-preserving sector* + corrección de lectura).

### Mapeo de Capítulos y Secciones Experimentales (Parte VII)
| Capítulo / § | Fenómeno Físico | Identidad Métrica $qg$ | Falsación & Reducción de Ruido |
|---|---|---|---|
| Cap. 28 (§151) | Oscilación en vacío (3 sabores, 3 qubits) | $qg_Z = \cos^2(2\theta) + \sin^2(2\theta)\cos\Delta$; $\theta = \frac{1}{4}\arccos(qg_Z)$ en el máximo | Filtro $qg$ reduce error $\sim 3.5\times$; dispersión independiente de $\theta$ |
| Cap. 29 (§152) | Efecto de materia terrestre (MSW) | Ruptura aparente de simetría CP generada por densidad electrónica | Materia imita violación CP ($A_{CP} = 0.042$ con $\delta_{CP}=0$). Circuito filtrado la aísla |
| Cap. 30 (§154) | Oscilaciones colectivas (gas denso de $\nu$) | Conservación de número de $\nu_2$ en base de masa ($H_{int} \propto \sum \sigma_i \cdot \sigma_j$) | Conservación estricta de peso en base de masa; ganancia de filtro $2.0\text{--}2.8\times$ |
| Cap. 31 (§156) | Conversión adiabática solar (MSW) | $P_{ee}$ es estrictamente lineal en $qg_Z$ del qubit $\nu_2$ | Circuito en capas $r=8$ filtra error ($2.0\times$); circuito compilado mono-rotacional le gana |
| Cap. 32 (§163) | Auditoría: Jerarquía, fases Majorana y $\nu$ estéril | Sensibilidad a signo $\Delta m^2_{31}$ e invariancia gauge de fases | Fases de Majorana son matemáticamente invisibles en oscilación; $\nu$ estéril añade 4to qubit |

## 1. Instalación del Entorno Cuántico (`qang 0.14.0`)

In [ ]:
import importlib, subprocess, sys
# Instalación del ecosistema qang y backends de simulación calibrada
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "qiskit-aer", "qiskit-ibm-runtime", "matplotlib"], check=False)
importlib.invalidate_caches()

import qang
from qang import neutrino as nu
from qang import ibm
from qang import formulation as F
from qang import integrations

print(f"Framework cuántico cargado: qang v{qang.__version__}")

## 2. Configuración de Hardware Calibrado y Protocolo de Filtrado

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import transpile
from qiskit.quantum_info import Statevector

BACKENDS = ("FakeBrisbane", "FakeTorino")   # Arquitecturas nativas ECR y CZ
FR = "filtered_readout"
T0 = time.time()

def get_sampler(backend):
    try:
        from qiskit_ibm_runtime.executor_sampler import Sampler
    except ImportError:
        from qiskit_ibm_runtime import SamplerV2 as Sampler
    return Sampler(mode=backend)

def execute_qg_circuits(circuits, b, weight, shots=4000):
    ts = [transpile(qc, b, optimization_level=1, seed_transpiler=137) for qc in circuits]
    sampler = get_sampler(b)
    res = sampler.run(ts, shots=shots).result()
    return [ibm.ibm_qg(r, t, b, weight=weight) for r, t in zip(res, ts)]

## 3. Oscilaciones en Vacío: Invarianza de Dispersión y 3 Sabores (Cap. 28, §151)
En el espacio $qg$, para un sistema de 2 sabores sobre un qubit:
$$qg_Z = 1 - 2P(e \to \mu) = \cos^2(2\theta) + \sin^2(2\theta)\cos\Delta$$
En el máximo de oscilación ($\Delta = \pi$), el ángulo de mezcla satisface la biyección $\theta = \frac{1}{4}\arccos(qg_Z)$, heredando una varianza de disparo completamente uniforme respecto de $\theta$.

In [ ]:
rng = np.random.default_rng(151)
N_SHOTS = 1000
print(f"Ángulo θ    Media θ̂       Desv. Est.    Teoría ({nu.mixing_angle_std(N_SHOTS):.4f})")
for th in (0.1, 0.3, 0.5, 0.7):
    k = rng.binomial(N_SHOTS, nu.two_flavour_probability(th, math.pi), 2000)
    est = np.array([nu.mixing_angle_from_maximum(1 - 2 * x / N_SHOTS) for x in k])
    print(f" {th:4.1f}      {est.mean():.4f}        {est.std(ddof=1):.4f}        {nu.mixing_angle_std(N_SHOTS):.4f}")

### Tres sabores en tres qubits bajo preservación de peso de Hamming (sector $w=1$)
La matriz PMNS descompone en 3 rotaciones RBS y 2 compuertas de fase locales. La propagación preserva estrictamente el peso de excitación.

In [ ]:
L, DCP = 1300.0, -math.pi / 2 # Línea de base tipo DUNE
ENERGIES = (0.8, 1.2, 1.6, 2.0, 2.6, 3.2, 4.0, 5.0)
exact = np.array([nu.three_flavour_probabilities(L, E, DCP)[1] for E in ENERGIES])
print("Probabilidades exactas P(mu -> e, mu, tau) a 2.6 GeV:", np.round(nu.three_flavour_probabilities(L, 2.6, DCP)[1], 3))
print(f"Asimetría CP exacta A_CP a 2.6 GeV: {nu.cp_asymmetry(L, 2.6, DCP):.4f}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(ENERGIES, exact[:, 1], "k-", lw=1.8, label="Exacto P(mu->mu)")

for name in BACKENDS:
    b = getattr(fp, name)()
    outs = execute_qg_circuits([nu.three_flavour_circuit("mu", L, E, DCP) for E in ENERGIES], b, 1, 2000)
    err = {k: np.abs(np.array([ibm.occupations(o, k) for o in outs]) - exact).mean() for k in ("raw", FR)}
    oc = execute_qg_circuits([nu.three_flavour_circuit("mu", L, 2.6, DCP, anti) for anti in (False, True)], b, 1, 10000)
    acp = {k: ibm.occupations(oc[0], k)[0] - ibm.occupations(oc[1], k)[0] for k in ("raw", FR)}
    print(f"{name:<12}: Error medio crudo {err['raw']:.3f} | Filtrado qg {err[FR]:.3f} (Ganancia {err['raw']/err[FR]:.1f}x) | A_CP {acp[FR]:.4f}")
    ax.plot(ENERGIES, [ibm.occupations(o, "raw")[1] for o in outs], "x--", alpha=0.7, label=f"{name} crudo")
    ax.plot(ENERGIES, [ibm.occupations(o, FR)[1] for o in outs], "o:", lw=1.5, label=f"{name} filtrado qg")

ax.set_xlabel("E (GeV)")
ax.set_ylabel("P(mu -> mu)")
ax.set_title("Oscilación 3-sabores con Filtro de Simetría qg")
ax.legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()

## 4. Efecto de Materia Terrestre: Falsación de Violación CP (Cap. 29, §152)
El potencial de fondo de electrones $A = 2\sqrt{2} G_F N_e E$ genera un término asimétrico que imita violación CP ($A_{CP} = 0.042$ a $\delta_{CP}=0$). El filtro $qg$ aísla la señal física del ruido de despolarización de los qubits.

In [ ]:
RHO, R = 2.85, 2
acp_exact = nu.cp_asymmetry(L, 2.6, 0.0, rho=RHO)
sv = lambda anti: np.abs(Statevector(nu.matter_trotter_circuit("mu", L, 2.6, 0.0, RHO, R, antineutrino=anti, measure=False)).data[1]) ** 2
print(f"A_CP (delta_CP = 0): Exacto = {acp_exact:.4f} | Trotter r={R} sin ruido = {sv(False) - sv(True):.4f}")

for name in BACKENDS:
    b = getattr(fp, name)()
    oc = execute_qg_circuits([nu.matter_trotter_circuit("mu", L, 2.6, 0.0, RHO, R, antineutrino=anti) for anti in (False, True)], b, 1, 10000)
    acp = {k: ibm.occupations(oc[0], k)[0] - ibm.occupations(oc[1], k)[0] for k in ("raw", FR)}
    print(f"{name:<12}: A_CP crudo {acp['raw']:.4f} | Filtrado qg {acp[FR]:.4f}")

## 5. Oscilaciones Colectivas en Medios Ultra-densos (Cap. 30, §154)
En supernovas, la auto-interacción $\nu$-$\nu$ conserva el número de excitaciones en base de masa ($N_{\nu_2}$), no en la de sabor. El filtro $qg$ aplicado en el sector de peso 2 suprime el ruido conservando la correlación cuántica.

In [ ]:
OMEGAS, MU, THETA, DT = (0.25, 0.5, 0.75, 1.0), 4.0, 0.195, 0.5
TIMES = (0.5, 1.0, 1.5, 2.0)
circ = lambda t, m=True: nu.collective_trotter_circuit(OMEGAS, MU, THETA, "0011", t, int(round(t / DT)), "mass", "mass", measure=m)
idx = np.arange(16)
ref = []
for t in TIMES:
    p = Statevector(circ(t, False)).probabilities()
    ref.append([p[(idx >> i) & 1 == 1].sum() for i in range(4)])
ref = np.array(ref)

for name in BACKENDS:
    b = getattr(fp, name)()
    outs = execute_qg_circuits([circ(t) for t in TIMES], b, 2, 4000)
    err = {k: np.abs(np.array([ibm.occupations(o, k) for o in outs]) - ref).mean() for k in ("raw", FR)}
    print(f"{name:<12}: Error medio P(nu_2) crudo {err['raw']:.3f} | Filtrado qg {err[FR]:.3f} (Ganancia {err['raw'] / err[FR]:.1f}x) | Disparos retenidos: {np.mean([o['kept_raw'] for o in outs]):.2f}")

## 6. Conversión Adiabática Solar MSW (Cap. 31, §156)
La supervivencia $P_{ee}$ solar resulta estrictamente lineal en $qg_Z$ del modo $\nu_2$. Se demuestra la auditoría de falsación: el circuito compilado analíticamente supera en fidelidad al circuito por capas Trotter, evidenciando el límite de coherencia en observables de población.

In [ ]:
H, X, RS = 100.0, 800.0, 8
E_SOL = (0.0005, 0.001, 0.002, 0.004, 0.006, 0.010, 0.015)   # GeV
pee = lambda p2s: np.array([nu.solar_pee_from_p2(float(p)) for p in p2s])
ex = pee([nu.solar_p2_exact(E, H, X) for E in E_SOL])
slab = pee([nu.solar_p2(E, H, X, RS) for E in E_SOL])

print("Exacto P_ee:      ", np.round(ex, 3))
print("Adiabático P_ee:  ", np.round([nu.solar_pee_adiabatic(E) for E in E_SOL], 3))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.semilogx(np.array(E_SOL) * 1e3, ex, "k-", lw=2, label="Exacto")

for name in BACKENDS:
    b = getattr(fp, name)()
    so = execute_qg_circuits([nu.solar_circuit(E, H, X, RS) for E in E_SOL], b, 1, 4000)
    co = execute_qg_circuits([nu.solar_circuit(E, H, X, RS, compiled=True) for E in E_SOL], b, 1, 4000)
    est_slab_f = pee([ibm.occupations(o, FR)[1] for o in so])
    est_comp_f = pee([ibm.occupations(o, FR)[1] for o in co])
    print(f"{name:<12} | Error Trotter r=8 filtrado: {np.abs(est_slab_f - slab).mean():.4f} | Error Compilado: {np.abs(est_comp_f - slab).mean():.4f}")
    ax.semilogx(np.array(E_SOL) * 1e3, est_slab_f, "o:", label=f"{name} Capas r=8 (Filtro qg)")
    ax.semilogx(np.array(E_SOL) * 1e3, est_comp_f, "^--", label=f"{name} Compilado RBS")

ax.set_xlabel("E (MeV)")
ax.set_ylabel("P_ee")
ax.set_title("Transición Adiabática MSW en el Sol")
ax.legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()

## 7. Falsación y Límites de Observabilidad: Fases de Majorana y Neutrinos Estériles (Cap. 32, §163)
* **Fases de Majorana:** Se cancelan estrictamente en la evolución $U D U^\dagger$ de oscilaciones. Los circuitos transpilados con y sin fases son topológicamente idénticos.
* **Neutrinos Estériles (3+1):** Se proyecta como un 4to qubit en el subespacio de peso 1.

In [ ]:
L163, E163, RHO163 = 1300.0, 2.6, 2.85
# Verificación de invisibilidad de fases de Majorana en oscilación
inv = nu.majorana_invisible(L163, E163, -1.57, 1.0, 2.5, rho=RHO163)
print(f"Variación por fases de Majorana en probabilidad de oscilación: {inv:.2e} (Cero dentro de precisión numérica)")

b = getattr(fp, BACKENDS[0])()
m = [transpile(nu.majorana_circuit("mu", L163, E163, -1.57, a, c), b, optimization_level=1, seed_transpiler=137)
     for a, c in ((0, 0), (math.pi / 2, 3 * math.pi / 4))]
print(f"Circuitos cuánticos con y sin fase de Majorana son idénticos tras transpilación: {m[0] == m[1]}")

# Neutrino estéril (3+1)
ES163 = (0.2, 0.3, 0.4, 0.6, 0.9, 1.4)
ST = {"theta14": 0.15, "theta24": 0.15, "theta34": 0.0}
print("P(mu -> mu) con neutrino estéril (3+1) exacto a 0.5 km:", [round(float(nu.sterile_probabilities(0.5, E, **ST)[1, 1]), 3) for E in ES163])

## Conclusiones del Cuaderno y Conexión con el Libro
1. **Preservación de Peso:** El álgebra $qg$ demuestra que los circuitos de oscilación neutrónica conservan el peso de Hamming, permitiendo descartar disparos espurios y mitigar el ruido en dispositivos NISQ entre $2\times$ y $5\times$.
2. **Auditoría Teórica:** Se valida que la naturaleza adiabática de la conversión solar puede resolverse analíticamente con rotaciones RBS óptimas, y que la búsqueda de física más allá del Modelo Estándar (fases Majorana) requiere canales de desintegración ($etaeta0
u$) y no experimentos de oscilación en circuitos cuánticos.
3. **Framework `qang 0.14.0`:** La interoperabilidad nativa con Qiskit permite reproducir fielmente la Parte VII de la obra.